In [1]:
# On the Capella Cluster start an Interactive SLURM Job with: 
# 
# srun --pty --ntasks=1 --cpus-per-task=8 --gres=gpu:4 --time=10:00:00 --mem-per-cpu=16384 --nodes=1 --licenses=cat --account=p_scads_social_media bash -l
# 
# The command requests 1 task with 4 Nvidea-H100 GPUs for 10h

In [ ]:
import os
os.environ["HF_HOME"] = "/data/cat/ws/thha024h-masterthesis/hf_models"
import json
import re
import pandas as pd
from pathlib import Path
from vllm import LLM, SamplingParams

In [ ]:
# PATH SPECIFICATION
DATA   = "data/full.csv.zip"

PROMPT_PATH = "prompt-populism.md"
MODEL  = "hf_models/Qwen3-235B-A22B-Instruct-2507-FP8/"

In [ ]:
# LOAD DATASET
d = pd.read_csv(DATA)

# WHEN TESTING THE SCRIPT
#d = d.head(50)

# read in prompt
with open(PROMPT_PATH, "r") as f:
    PROMPT = f.read()

In [ ]:
# LOAD MODEL 
llm_kwargs = dict(
    model=MODEL,
    tensor_parallel_size=4 # use all 4 requested GPUs to load the Model that needs +50GB RAM 
)

sampling_params = SamplingParams(
    temperature=0.2, # temperature 0 prohibits effective reasoning
    top_p=0.9,
    max_tokens=4048  # allows long reasoning processes
)
llm = LLM(**llm_kwargs)
tokenizer = llm.get_tokenizer()

to_annotate = d[["id", "text"]].reset_index(drop=True)

In [ ]:
# PARSE TEXTS
STR_KEYS = [
    "holistic_redescription",
    "social_actors_analysis",
    "populist_explanation",
    "elitist_explanation",
    "intensity_explanation",
]
NUM_KEYS = [
    "populist_score",
    "elitist_score",
    "intensity_score",
]
d_results = []
for row, output in zip(rows_list, outputs):
    answer = output.outputs[0].text

    # --- 1. Attempt Standard JSON Parsing ---
    try:
        json_str = answer[answer.find("{"):answer.rfind("}") + 1]
        answer_data = json.loads(json_str)
    except (json.JSONDecodeError, ValueError, AttributeError):
        # --- 2. Regex Fallback ---
        try:
            parsed = {}

            for key in STR_KEYS:
                match = re.search(
                    rf'"{key}"\s*:\s*"(.*?)"\s*(?:,|}}\s*$)',
                    answer,
                    re.DOTALL,
                )
                if match:
                    parsed[key] = match.group(1).replace('"', "'")
                else:
                    parsed[key] = None

            for key in NUM_KEYS:
                match = re.search(rf'"{key}"\s*:\s*(-?\d+)', answer)
                if match:
                    parsed[key] = int(match.group(1))
                else:
                    parsed[key] = None

            answer_data = parsed
        except Exception as regex_error:
            answer_data = {
                "error": f"Parse Failed: {regex_error}",
                "raw_output": answer,
            }

    d_results.append({"id": row.id, **answer_data})

In [ ]:
# CONVERT ANNOTATIONS TO DATAFRAME AND LEFT_JOIN TO ORIGINAL
d_annotations = pd.DataFrame(d_results)
d_out = d.merge(d_annotations, on="id", how="left")

In [ ]:
# sanitycheck
d_annotations.head()

In [ ]:
# SAVE AS PARQUET
stem = Path(DATA).stem  # e.g. "bt_follow_2022-02-07_2022-02-14_tweets"
out_path = Path(DATA).parent / f"{stem}_annotated_populism.parquet"
d_out.to_parquet(out_path, index=False)
print(f"Saved {len(d_out)} rows to {out_path}")

In [ ]:
# After processing the Tweets copy data from the cluster via rsync
#
# rsync -avz thha024h@dataport1.hpc.tu-dresden.de:/data/cat/ws/thha024h-masterthesis/data/FILENAME Documents/synosys_masterthesis/